# CP2 Week 12 -- Package Hygiene

**Course:** Computer Programming 2 (CP2)
**Prerequisites:** Weeks 1-11
**Focus:** config module, stable paths, __init__.py, clean structure

## Learning Objectives
- Build a robust config module with stable paths
- Ensure code works on both Colab and local machines
- Clean up __init__.py exports
- Verify project structure is complete

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/ArifSolmaz/courseos-curriculum.git
# %cd courseos-curriculum/course-content

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Part 1: Stable Config Module

A good config module centralizes ALL settings and makes paths work regardless of where you run the code.

In [ ]:
import os

def get_config(base_dir=None):
    """Get project configuration with stable paths.
    
    Args:
        base_dir: Project root. Auto-detected if None.
    """
    if base_dir is None:
        for candidate in [".", "..", "../.."]:
            if os.path.exists(os.path.join(candidate, "src")):
                base_dir = candidate
                break
        else:
            base_dir = "."
    
    return {
        "project_name": "my_project",
        "track": "data",
        "version": "v2",
        "base_dir": os.path.abspath(base_dir),
        "raw_data_path": os.path.join(base_dir, "data", "raw", "sample.csv"),
        "cleaned_data_path": os.path.join(base_dir, "data", "cleaned", "cleaned.csv"),
        "report_path": os.path.join(base_dir, "reports", "report.json"),
        "figures_dir": os.path.join(base_dir, "reports", "figures"),
        "required_columns": ["id", "value"],
        "numeric_columns": ["value"],
        "value_ranges": {"value": (0, 100)},
        "threshold": 60,
    }

config = get_config()
print("Config:")
for k, v in config.items():
    print("  " + str(k) + ": " + str(v))

---
## Part 2: Project Structure Checklist

In [ ]:
def check_project_structure(base_dir="."):
    """Verify project has all required files and directories."""
    print("=== Project Structure Check ===\n")
    
    required = [
        "src",
        "data/raw",
        "data/cleaned",
        "reports",
        "reports/figures",
    ]
    
    for path in required:
        full = os.path.join(base_dir, path)
        exists = os.path.exists(full)
        status = "[OK]" if exists else "[MISSING]"
        print("  " + status + " " + path)
        if not exists:
            os.makedirs(full, exist_ok=True)
            print("       -> Created!")
    
    print("\nDone!")

check_project_structure()

---
## Part 3: Clean __init__.py

Your __init__.py should only export the PUBLIC API -- functions that notebooks and other code should use.

In [ ]:
# Example: a clean __init__.py
init_example = """
# Public API -- these are the only functions notebooks should use
from .config import get_config
from .io import load_data
from .cleaning import clean_data, validate_schema
from .analysis import analyze
from .plotting import plot
from .reporting import export_results, self_check

__all__ = [
    "get_config",
    "load_data",
    "clean_data",
    "validate_schema",
    "analyze",
    "plot",
    "export_results",
    "self_check",
]

__version__ = "2.0.0"
"""
print("Clean __init__.py:")
print(init_example)
print("__all__ tells Python which names are public.")
print("__version__ lets you track which version is running.")

---
## Part 4: Config Validation

Validate your config at startup to catch misconfigurations early.

In [ ]:
def validate_config(config):
    """Check that config has all required keys."""
    required_keys = [
        "project_name", "track", "version",
        "required_columns", "numeric_columns",
    ]
    
    missing = [k for k in required_keys if k not in config]
    if missing:
        raise ValueError(
            "Config missing required keys: " + str(missing) + "\n"
            "Available keys: " + str(list(config.keys()))
        )
    
    # Validate types
    if not isinstance(config["required_columns"], list):
        raise TypeError("required_columns must be a list")
    if not isinstance(config["numeric_columns"], list):
        raise TypeError("numeric_columns must be a list")
    
    print("Config validation passed: " + str(len(required_keys)) + " required keys present")
    return True

# Test: good config
config = get_config()
validate_config(config)

# Test: bad config
try:
    validate_config({"project_name": "test"})  # missing keys
except ValueError as e:
    print("\nBad config caught:")
    print(e)

**Expected Output:**
```
Config validation passed: 5 required keys present

Bad config caught:
Config missing required keys: ['track', 'version', 'required_columns', 'numeric_columns']
Available keys: ['project_name']
```

---
## Part 5: Loading Config from JSON

For more flexible projects, load config from a file.

In [ ]:
import json, os

def save_config(config, filepath="config/pipeline_config.json"):
    """Save config to JSON file."""
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    # Convert tuples to lists for JSON
    json_config = {}
    for k, v in config.items():
        if isinstance(v, dict):
            json_config[k] = {
                kk: list(vv) if isinstance(vv, tuple) else vv
                for kk, vv in v.items()
            }
        else:
            json_config[k] = v
    with open(filepath, "w") as f:
        json.dump(json_config, f, indent=2)
    print("Config saved: " + filepath)

def load_config(filepath="config/pipeline_config.json"):
    """Load config from JSON file."""
    with open(filepath) as f:
        config = json.load(f)
    # Convert value_ranges lists back to tuples
    if "value_ranges" in config:
        for k, v in config["value_ranges"].items():
            if isinstance(v, list) and len(v) == 2:
                config["value_ranges"][k] = tuple(v)
    print("Config loaded: " + filepath)
    return config

# Save and load
config = get_config()
save_config(config)
loaded = load_config()
print("Loaded config project:", loaded["project_name"])

**Expected Output:**
```
Config saved: config/pipeline_config.json
Config loaded: config/pipeline_config.json
Loaded config project: my_project
```

### Key Takeaway

- Config module = single source of truth for all settings
- Use os.path.join for cross-platform paths
- Auto-detect base_dir so code works on Colab and local
- Check project structure at startup to catch missing dirs
- Validate config to catch misconfigurations early
- Use __all__ in __init__.py to define the public API

---
## Homework: 12 Exercises

### Review (1-4)

In [ ]:
# HW1: Create get_config() for your project.


In [ ]:
# HW2: Use stable paths in all your modules.


In [ ]:
# HW3: Clean up __init__.py -- only export public functions.


In [ ]:
# HW4: Verify project structure with check_project_structure.


### Practice (5-8)

In [ ]:
# HW5: Add data validation to get_config (check files exist).


In [ ]:
# HW6: Support config override from environment variables.


In [ ]:
# HW7: Write a setup script that creates all needed directories.


In [ ]:
# HW8: Add a version string to your config.


### Challenge (9-11)

In [ ]:
# HW9: Load config from a JSON file instead of hard-coding.


In [ ]:
# HW10: Add config validation (required keys check).


In [ ]:
# HW11: Write a project health check that verifies everything.


### Mini-Project

In [ ]:
# HW12: Polish your project structure until check_project_structure
# and self_check both pass with zero failures.


---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)